In [ ]:
print("Mo")

In [ ]:
%pwd

In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# Extract text from PDF files in a directory
def load_pdf_from_directory(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf", 
        loader_cls=PyPDFLoader)

    documents = loader.load()
    return documents

In [ ]:
# import os

# os.chdir("/Users/mosalah/Build-a-Complete-Medical-Chatbot-")
# print(os.getcwd())
# print(os.path.exists("data"))
# print(os.listdir("data"))

In [ ]:
import os

print(os.getcwd())
print(os.listdir())

In [ ]:
extracted_data = load_pdf_from_directory(
    "/Users/mosalah/Build-a-Complete-Medical-Chatbot-/data"
)

In [ ]:
extracted_data


In [ ]:
len(extracted_data)

In [ ]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document
            (page_content=doc.page_content, metadata={"source": src}
        )
    )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
# print(type(minimal_docs))
# print(len(minimal_docs))
# print(minimal_docs[:1])

In [ ]:
#page_content=''

In [ ]:
# split the documents into chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    text_chunk = text_splitter.split_documents(minimal_docs)
    return text_chunk

In [ ]:
text_chunk = text_split(minimal_docs)
print(f"Number of chunks: {len(text_chunk)}")

In [ ]:
text_chunk

In [ ]:
import langchain
print(langchain.__version__)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

def create_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name= model_name
    )
    return embeddings

embeddings = create_embeddings()

In [ ]:
embeddings

In [ ]:
vector = embeddings.embed_query("What is the purpose of this document?")
vector

In [ ]:
print(f"Vector length: {len(vector)}")

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()


In [ ]:
#PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
#OPENAI_ENVIRONMENT_KEY = os.getenv("OPENAI_ENVIRONMENT_KEY")

#os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
#os.environ["OPENAI_ENVIRONMENT_KEY"] = OPENAI_ENVIRONMENT_KEY


PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
pc

In [ ]:
#pc.list_indexes()

In [ ]:
import pinecone
print(pinecone.__version__)

In [ ]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

# Hämta alla befintliga index
existing_indexes = pc.list_indexes().names()

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunk,
    embedding=embeddings,
    index_name=index_name
)

In [ ]:
# load existing index
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

# Add More Data to the Existing Pinecone Index

In [ ]:
dswith = Document(
    page_content="This is a sample document for testing.",
    metadata={"source": "sample_source.pdf"}
)

In [ ]:
docsearch.add_documents([dswith])

In [ ]:
retrieved = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
retrieved_docs = retrieved.invoke("What is Acne?")
retrieved_docs

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print("Current folder:", os.getcwd())
print("API key found:", os.getenv("OPENAI_API_KEY") is not None)

In [ ]:
from dotenv import dotenv_values

config = dotenv_values("/Users/mosalah/Build-a-Complete-Medical-Chatbot-/.env")

print(config)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("OpenAI:", os.getenv("OPENAI_API_KEY") is not None)
print("Pinecone:", os.getenv("PINECONE_API_KEY") is not None)

In [ ]:
#from langchain_openai import ChatOpenAI
#chatModel = ChatOpenAI(model="gpt-4o")

from langchain_ollama import ChatOllama

chatModel = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt =(
    "You are an medical assistant for question answering tasks."
    "Use the following pieces of retrieved documents context to answer"
    "the question. If you don't know the answer, say that you don't know."
    "Use three sentences maximum and keep the answer concise and clear."
    "\n\n"
    "(context)"
)

prompt = ChatPromptTemplate.from_messages(
    [
    ("system", system_prompt),
    ("human", "{input}")
    ]
)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
Use the following context to answer the question.

Context:
{context}

Question:
{input}

Answer:
"""
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retrieved, question_answer_chain)

In [ ]:
response = rag_chain.invoke({
    "input": "Explain both acromegaly and gigantism, and describe the main difference between them."
})

print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "what is Acne?"})
print(response["answer"])

In [ ]:
#from langchain_ollama import ChatOllama

#llm = ChatOllama(
 #   model="llama3.2:3b",
  #  temperature=0
#)

#response = llm.invoke("What are acromegaly and gigantism?")
#print(response.content)

In [ ]:
response = rag_chain.invoke({"input": "what is the treatment of Acne?"})
print(response["answer"])